In [ ]:
"""
M4
MODULE AMÉLIORÉ — CLASSIFICATEUR RISQUE CARDIOVASCULAIRE
Dataset : NIH CLAHE Enhanced Chest X-rays (112 000+ images)
Amélioration vs ancien : dataset équilibré 2000/2000/2000 au lieu de 1984/80/65
À coller dans un Kaggle Notebook avec GPU T4
"""
import os
import json
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_auc_score, roc_curve)
from PIL import Image

# ============================================================
# CONFIGURATION
# ============================================================
BASE_PATH = '/kaggle/input/datasets/rahulogoel/clahe-enhancement-on-chestx-ray14'
CSV_PATH  = f'{BASE_PATH}/Data_Entry.csv'
IMG_DIRS  = [f'{BASE_PATH}/IMAGE/IMAGE']  # Un seul dossier IMAGE au lieu de 12

# AMÉLIORATION PRINCIPALE : dataset équilibré
MAX_PAR_CLASSE = 2000   # 2000 par classe au lieu de 1984/80/65

CATEGORIES = ['Normal', 'Cardiomegaly', 'Edema']
EPOCHS     = 25
BATCH      = 16
LR         = 1e-4

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

# ============================================================
# 1. INDEXATION RAPIDE DES IMAGES
# ============================================================
def indexer_images(img_dirs):
    """
    Scanne tous les dossiers images_001 à images_012
    et crée un dictionnaire {nom_image: chemin_complet}
    """
    print("Indexation des images...")
    index = {}
    for img_dir in img_dirs:
        if not os.path.exists(img_dir):
            continue
        for fname in os.listdir(img_dir):
            if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                index[fname] = os.path.join(img_dir, fname)
    print(f"Images indexées : {len(index)}")
    return index

# ============================================================
# 2. CNN EfficientNet — IDENTIQUE AU MODULE 3 ORIGINAL
# ============================================================
class CardioRiskCNN(nn.Module):
    def __init__(self, num_classes=3, pretrained=True):
        super(CardioRiskCNN, self).__init__()

        self.backbone = models.efficientnet_b0(
            weights=models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        )

        orig = self.backbone.features[0][0]
        self.backbone.features[0][0] = nn.Conv2d(
            1, orig.out_channels,
            kernel_size=orig.kernel_size,
            stride=orig.stride,
            padding=orig.padding,
            bias=False
        )
        with torch.no_grad():
            self.backbone.features[0][0].weight = nn.Parameter(
                orig.weight.mean(dim=1, keepdim=True)
            )

        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(256, num_classes)
        )

        self._activations = None
        self.backbone.features[-1].register_forward_hook(self._save_activation)

    def _save_activation(self, module, input, output):
        self._activations = output

    def forward(self, x):
        return self.backbone(x)

    def get_gradcam(self, input_tensor, target_class=None):
        self.eval()
        input_tensor = input_tensor.clone().requires_grad_(True)
        self._activations = None
        output = self.backbone(input_tensor)
        if target_class is None:
            target_class = output.argmax(dim=1).item()
        self.zero_grad()
        output[0, target_class].backward()
        if self._activations is None:
            return None, output
        activations = self._activations.detach().cpu().numpy()[0]
        weights     = np.mean(activations, axis=(1, 2))
        cam         = np.zeros(activations.shape[1:], dtype=np.float32)
        for i, w in enumerate(weights):
            cam += w * activations[i]
        cam = np.maximum(cam, 0)
        if cam.max() > 0:
            cam /= cam.max()
        return cam, output

# ============================================================
# 3. DATASET AMÉLIORÉ — CSV + INDEX IMAGES
# ============================================================
class CardioRiskDataset(Dataset):
    def __init__(self, csv_path, img_index, max_par_classe=2000, transform=None):
        self.transform   = transform
        self.image_paths = []
        self.labels      = []

        df = pd.read_csv(csv_path)
        print(f"CSV chargé : {len(df)} lignes")

        compteurs = Counter()
        ignores   = 0

        for _, row in df.iterrows():
            img_name   = str(row['Image Index'])
            labels_str = str(row['Finding Labels'])

            # Lookup O(1) dans l'index
            if img_name not in img_index:
                ignores += 1
                continue

            # Label depuis CSV
            label = self._label_depuis_csv(labels_str)
            if label is None:
                continue

            # AMÉLIORATION : limite par classe pour équilibre
            cat = CATEGORIES[label]
            if compteurs[cat] >= max_par_classe:
                continue

            self.image_paths.append(img_index[img_name])
            self.labels.append(label)
            compteurs[cat] += 1

            # Arrêter si toutes les classes sont pleines
            if all(compteurs[c] >= max_par_classe for c in CATEGORIES):
                break

        print(f"Images ignorées (non trouvées) : {ignores}")
        print(f"\nDistribution finale :")
        for cat in CATEGORIES:
            print(f"  {cat:<15}: {compteurs[cat]} images")

        # Poids pour WeightedRandomSampler
        total = len(self.labels)
        counts = Counter(self.labels)
        self.sample_weights = [
            total / (counts[l] * len(CATEGORIES))
            for l in self.labels
        ]

    def _label_depuis_csv(self, labels_str):
        if 'Cardiomegaly' in labels_str:
            return 1
        elif 'Edema' in labels_str:
            return 2
        elif 'No Finding' in labels_str:
            return 0
        return None

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img   = Image.open(self.image_paths[idx]).convert('L')
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

# ============================================================
# 4. ÉVALUATION COMPLÈTE
# ============================================================
def evaluate_on_test(model, test_loader, device):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs   = imgs.to(device)
            output = model(imgs)
            probs  = torch.softmax(output, dim=1).cpu().numpy()
            preds  = output.argmax(dim=1).cpu().numpy()
            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    all_probs  = np.array(all_probs)
    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)

    print("\n" + "=" * 55)
    print("  RAPPORT TEST SET")
    print("=" * 55)
    print(classification_report(all_labels, all_preds,
                                target_names=CATEGORIES, digits=3))

    try:
        auc = roc_auc_score(all_labels, all_probs,
                            multi_class='ovr', average='macro')
        print(f"AUC-ROC macro : {auc:.4f}")
    except Exception as e:
        auc = None

    # Matrice de confusion
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    cm = confusion_matrix(all_labels, all_preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CATEGORIES, yticklabels=CATEGORIES, ax=axes[0])
    axes[0].set_title('Matrice de Confusion', fontweight='bold')
    axes[0].set_ylabel('Vrai label')
    axes[0].set_xlabel('Prédit')

    for i, (cat, color) in enumerate(zip(CATEGORIES, ['green','red','orange'])):
        binary = (all_labels == i).astype(int)
        if binary.sum() > 0:
            fpr, tpr, _ = roc_curve(binary, all_probs[:, i])
            auc_i = roc_auc_score(binary, all_probs[:, i])
            axes[1].plot(fpr, tpr, color=color,
                        label=f'{cat} (AUC={auc_i:.2f})', linewidth=2)
    axes[1].plot([0,1],[0,1],'k--', alpha=0.5)
    axes[1].set_title('Courbes ROC', fontweight='bold')
    axes[1].set_xlabel('Taux faux positifs')
    axes[1].set_ylabel('Taux vrais positifs')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    correct  = (all_preds == all_labels)
    maxprobs = all_probs.max(axis=1)
    axes[2].hist(maxprobs[correct],  bins=20, alpha=0.7, color='green',
                label=f'Correct (n={correct.sum()})')
    axes[2].hist(maxprobs[~correct], bins=20, alpha=0.7, color='red',
                label=f'Incorrect (n={(~correct).sum()})')
    axes[2].set_title('Confiance du modèle', fontweight='bold')
    axes[2].set_xlabel('Score de confiance')
    axes[2].legend(); axes[2].grid(True, alpha=0.3)

    plt.suptitle('Évaluation Module 3 — Dataset NIH équilibré',
                fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('module3_evaluation.png', dpi=150, bbox_inches='tight')
    plt.show()

    return round((all_preds == all_labels).mean(), 4)

# ============================================================
# 5. ENTRAÎNEMENT
# ============================================================
if __name__ == "__main__":

    # Indexation des images
    img_index = indexer_images(IMG_DIRS)

    t_train = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5])
    ])
    t_val = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5])
    ])

    # Dataset équilibré
    dataset = CardioRiskDataset(
        csv_path      = CSV_PATH,
        img_index     = img_index,
        max_par_classe= MAX_PAR_CLASSE,
        transform     = t_train
    )

    n       = len(dataset)
    n_train = int(0.70 * n)
    n_val   = int(0.15 * n)
    n_test  = n - n_train - n_val
    print(f"\nSplit : Train={n_train} | Val={n_val} | Test={n_test}")

    train_set, val_set, test_set = random_split(
        dataset, [n_train, n_val, n_test],
        generator=torch.Generator().manual_seed(42)
    )
    val_set.dataset.transform  = t_val
    test_set.dataset.transform = t_val

    # WeightedRandomSampler
    train_w = [dataset.sample_weights[i] for i in train_set.indices]
    sampler  = torch.utils.data.WeightedRandomSampler(
        train_w, num_samples=len(train_w), replacement=True
    )

    train_loader = DataLoader(train_set, batch_size=BATCH,
                              sampler=sampler, num_workers=2)
    val_loader   = DataLoader(val_set,   batch_size=BATCH,
                              shuffle=False, num_workers=2)
    test_loader  = DataLoader(test_set,  batch_size=BATCH,
                              shuffle=False, num_workers=2)

    # Class weights dans la loss
    counts = [dataset.labels.count(i) for i in range(len(CATEGORIES))]
    total  = sum(counts)
    cw     = torch.tensor([total / (c * len(CATEGORIES)) for c in counts],
                          dtype=torch.float).to(device)
    print(f"Class weights : { {cat: round(w,2) for cat,w in zip(CATEGORIES,cw.tolist())} }")

    model     = CardioRiskCNN(num_classes=len(CATEGORIES)).to(device)
    criterion = nn.CrossEntropyLoss(weight=cw)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    best_val_acc = 0.0
    historique   = []

    print(f"\n{'Époque':<8} {'Loss':<12} {'Train Acc':<12} {'Val Acc':<10}")
    print("-" * 44)

    for epoch in range(EPOCHS):
        model.train()
        t_loss, correct, total_s = 0, 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            out  = model(imgs)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
            t_loss  += loss.item()
            correct += (out.argmax(1) == labels).sum().item()
            total_s += labels.size(0)

        train_acc = correct / total_s

        model.eval()
        v_correct, v_total = 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                out       = model(imgs)
                v_correct += (out.argmax(1) == labels).sum().item()
                v_total   += labels.size(0)

        val_acc = v_correct / v_total
        scheduler.step()

        historique.append({
            'epoch':      epoch + 1,
            'train_loss': round(t_loss / len(train_loader), 4),
            'train_acc':  round(train_acc, 4),
            'val_acc':    round(val_acc, 4)
        })

        marker = "  ← meilleur" if val_acc > best_val_acc else ""
        print(f"{epoch+1:<8} {t_loss/len(train_loader):.4f}{'':7} "
              f"{train_acc:.3f}{'':7} {val_acc:.3f}{marker}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), 'cardio_risk_v2_best.pth')

    torch.save(model.state_dict(), 'cardio_risk_v2_final.pth')
    with open('historique_module3_v2.json', 'w') as f:
        json.dump(historique, f, indent=2)

    print(f"\nMeilleure Val Acc : {best_val_acc:.3f}")

    # Courbes
    eps    = [x['epoch']      for x in historique]
    losses = [x['train_loss'] for x in historique]
    t_accs = [x['train_acc']  for x in historique]
    v_accs = [x['val_acc']    for x in historique]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].plot(eps, losses, 'b-o', markersize=3, label='Train Loss')
    axes[0].set_title('Loss', fontweight='bold')
    axes[0].set_xlabel('Époques')
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(eps, t_accs, 'b-o', markersize=3, label='Train Acc')
    axes[1].plot(eps, v_accs, 'r-o', markersize=3, label='Val Acc')
    axes[1].set_title('Accuracy', fontweight='bold')
    axes[1].set_xlabel('Époques')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    plt.suptitle('Module 3 v2 — NIH équilibré 2000/2000/2000',
                fontweight='bold')
    plt.tight_layout()
    plt.savefig('courbes_module3_v2.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Évaluation finale
    model.load_state_dict(torch.load('cardio_risk_v2_best.pth',
                                     map_location=device))
    test_acc = evaluate_on_test(model, test_loader, device)
    print(f"Test Accuracy : {test_acc}")

    print("\nFichiers générés :")
    print("  cardio_risk_v2_best.pth  ← à télécharger sur Drive")
    print("  module3_evaluation.png")
    print("  courbes_module3_v2.png")

# ============================================================
# SAUVEGARDE AUTOMATIQUE — KAGGLE
# ============================================================
import os

SAVE_DIR = '/kaggle/working/'

fichiers = [
    'cardio_risk_v2_best.pth',
    'cardio_risk_v2_final.pth',
    'historique_module3_v2.json',
    'module3_evaluation.png',
    'courbes_module3_v2.png',
]

print("\n=== FICHIERS SAUVEGARDÉS DANS /kaggle/working/ ===")
for f in fichiers:
    path = os.path.join(SAVE_DIR, f)
    if os.path.exists(path):
        taille = os.path.getsize(path) / 1024 / 1024
        print(f"  ✓ {f} ({taille:.1f} MB)")
    else:
        print(f"  ✗ {f} non trouvé")

print("\nPour télécharger :")
print("  Kaggle → Output (panneau droit) → clic sur le fichier → Download")